# Streamlined: SED Fitting with Bagpipes

Fit the broadband photometry of the AGEL0206 deflector galaxy using
[Bagpipes](https://bagpipes.readthedocs.io/) with an exponential-tau star formation
history and Calzetti dust attenuation.

**Filters:** HST F200LP, HST F140W, JWST F150W2, JWST F322W2  
**Redshift:** z = 0.675 (spectroscopic, narrow prior 0.674--0.676)  
**Key result:** log(M*/Msun) = 11.34 +0.06/-0.08

This is a clean version of `02_Bagpipes_SED_fitting.ipynb` with exploratory
cells and empty placeholders removed.

## 1. Imports and setup

In [ ]:
import numpy as np
import bagpipes as pipes
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
from astropy.constants import c
from astropy.io import fits, ascii

%matplotlib inline

# Plotting style
plt.rcParams['figure.facecolor'] = 'white'
plt.rc('font', family='serif', size=15)
plt.rc('axes', linewidth=1.5, labelsize=20)
plt.rc('xtick', labelsize=22, direction='in')
plt.rc('xtick.major', size=10, pad=4)
plt.rc('xtick.minor', size=5, pad=4)
plt.rc('ytick', labelsize=22, direction='in')
plt.rc('ytick.major', size=10)
plt.rc('ytick.minor', size=5)
plt.rc('legend', fontsize=15)

## 2. Input photometry

AB magnitudes measured from HST WFC3 (F200LP, F140W) and JWST NIRCam (F150W2, F322W2)
imaging using the interactive aperture photometry scripts in `scripts/`.
Convert to F_lambda flux density for Bagpipes.

In [ ]:
# Measured AB magnitudes and errors from aperture photometry
abmags = np.array([22.6126, 19.1335, 18.9425, 18.6042]) * u.ABmag
abmag_errors = np.array([0.0172, 0.0044, 0.0005, 0.0003]) * u.ABmag
filter_names = ['F200LP', 'F140W', 'F150W2', 'F322W2']
pivot_wavelengths = np.array([4971.9, 13923, 16720, 32470]) * u.AA

# Convert AB magnitudes to F_lambda flux density
fnu_units = u.erg / u.s / u.Hz / u.cm**2
flam_units = u.erg / (u.s * u.cm**2 * u.AA)

fluxes_nu = abmags.to(fnu_units)
fluxes_lambda = (fluxes_nu * (c / pivot_wavelengths**2)).to(flam_units)

# Propagate magnitude errors to flux errors (asymmetric in flux space)
fnu_err_upper = ((abmags - abmag_errors).to(fnu_units)) - fluxes_nu
fnu_err_lower = fluxes_nu - ((abmags + abmag_errors).to(fnu_units))
flam_err_upper = (fnu_err_upper * (c / pivot_wavelengths**2)).to(flam_units)
flam_err_lower = (fnu_err_lower * (c / pivot_wavelengths**2)).to(flam_units)
fluxes_lambda_error = np.maximum(flam_err_upper, flam_err_lower)

# Plot the SED
plt.figure(figsize=(10, 6))
plt.errorbar(pivot_wavelengths.to(u.micron).value, fluxes_lambda.value,
             yerr=fluxes_lambda_error.value, fmt='o', capsize=5, markersize=8)
for i, name in enumerate(filter_names):
    plt.annotate(name, (pivot_wavelengths.to(u.micron).value[i], fluxes_lambda.value[i]),
                 textcoords="offset points", xytext=(10, 10), fontsize=12)
plt.yscale('log')
plt.xlabel('Wavelength (micron)')
plt.ylabel(r'F$_\lambda$ (erg/s/cm$^2$/Ang)')
plt.title('AGEL0206 Deflector Photometry')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Load filter transmission curves

In [ ]:
# Load filter transmission curves from .dat files
# Note: These are in the repo root directory
hst200 = ascii.read('HST_WFC3_UVIS1.F200LP.dat')
hst140 = ascii.read('HST_WFC3_IR.F140W.dat')
jwst150 = ascii.read('JWST_NIRCam.F150W2.dat')
jwst322 = ascii.read('JWST_NIRCam.F322W2.dat')  # Note: file is NIRCam not NIRCAM

filters = [hst200, hst140, jwst150, jwst322]

plt.figure(figsize=(10, 6))
for filt, name in zip(filters, filter_names):
    plt.plot(filt['col1'], filt['col2'], label=name)
plt.yscale('log')
plt.xlabel('Wavelength (Angstrom)')
plt.ylabel('Transmission')
plt.title('Filter Transmission Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Set up Bagpipes galaxy object

Define the photometry loading function and create the Bagpipes galaxy object.
We use 10% fractional errors as a conservative estimate for the fit.

In [ ]:
# Use 10% fractional errors (conservative, accounts for systematics)
fluxes_lambda_error_fit = np.array([0.1, 0.1, 0.1, 0.1]) * fluxes_lambda

def load_data(ID):
    """Return photometry array for Bagpipes: [flux, error] per filter."""
    return np.array([fluxes_lambda.value, fluxes_lambda_error_fit.value]).T

# Filter list must match the .dat files in the working directory
filt_list = [
    'HST_WFC3_UVIS1.F200LP.dat',
    'HST_WFC3_IR.F140W.dat',
    'JWST_NIRCam.F150W2.dat',
    'JWST_NIRCam.F322W2.dat'  # Case must match actual filename on Linux
]

# Create the Bagpipes galaxy object
galaxy = pipes.galaxy("0206_real", load_data, spectrum_exists=False,
                       filt_list=filt_list, phot_units='ergscma')
fig = galaxy.plot()

## 5. Define the SED model

Exponential-tau star formation history with Calzetti dust attenuation.
The redshift is tightly constrained from spectroscopy (z = 0.674--0.676).

In [ ]:
# Exponential-tau star formation history
exp = {
    "age": (0.1, 15.),          # Galaxy age: 100 Myr to 15 Gyr
    "tau": (0.3, 10.),          # e-folding time: 300 Myr to 10 Gyr
    "massformed": (1., 15.),    # log10(M_formed / M_sun): 1 to 15
    "metallicity": (0., 2.5)    # Metallicity: 0 to 2.5 Z_solar
}

# Calzetti dust attenuation
dust = {
    "type": "Calzetti",
    "Av": (0., 2.)              # V-band extinction: 0 to 2 mag
}

# Full model specification
fit_instructions = {
    "redshift": (0.674, 0.676),  # Narrow prior from spectroscopic redshift
    "exponential": exp,
    "dust": dust
}

## 6. Run the fit

Uses MultiNest (preferred) or Nautilus (fallback) as the Bayesian sampler.
Results are cached in `pipes/posterior/` — delete the `.h5` file to re-run.

In [ ]:
# Run the Bagpipes fit
fit = pipes.fit(galaxy, fit_instructions, run="AGEL0206")

try:
    fit.fit(verbose=True, sampler="multinest", n_live=400)
except (AttributeError, OSError) as e:
    print(f"MultiNest error: {e}")
    print("Falling back to nautilus sampler...")
    fit.fit(verbose=True, sampler="nautilus", n_live=400)

## 7. Results

In [ ]:
# Extract key posterior quantities
samples = fit.posterior.samples

# Stellar mass
median_Mstellar = np.median(samples["stellar_mass"])
sigma_Mstellar = np.abs(median_Mstellar - np.percentile(samples["stellar_mass"], [16, 84]))
print(f"Stellar mass: log(M*/Msun) = {median_Mstellar:.2f} +{sigma_Mstellar[1]:.2f} -{sigma_Mstellar[0]:.2f}")

# Mass-weighted age
age_16, age_84 = np.percentile(samples["mass_weighted_age"], (16, 84))
print(f"Mass-weighted age: {np.median(samples['mass_weighted_age']):.1f} [{age_16:.1f} - {age_84:.1f}] Gyr")

# Specific star formation rate
median_ssfr = np.median(np.log10(samples["sfr"]) - samples["stellar_mass"])
print(f"log(sSFR) = {median_ssfr:.2f} yr^-1")

In [ ]:
# Generate diagnostic plots: SED, SFH, and corner plot
fit.posterior.get_advanced_quantities()

fig = fit.plot_spectrum_posterior(save=False, show=True)
fig = fit.plot_sfh_posterior(save=False, show=True)
fig = fit.plot_corner(save=False, show=True)

In [ ]:
# Summary figure: observed + model photometry with 1D posterior histograms
fig = plt.figure(figsize=(12, 7))
gs = mpl.gridspec.GridSpec(7, 4, hspace=3., wspace=0.1)

# Top panel: photometry comparison
ax1 = plt.subplot(gs[:4, :])
pipes.plotting.add_observed_photometry(fit.galaxy, ax1, zorder=10)
pipes.plotting.add_photometry_posterior(fit, ax1)

# Bottom panels: 1D posterior histograms
labels = ["sfr", "mass_weighted_age", "stellar_mass", "ssfr"]
for i in range(4):
    ax = plt.subplot(gs[4:, i])
    pipes.plotting.hist1d(fit.posterior.samples[labels[i]], ax, smooth=True, label=labels[i])

plt.show()